# 12 - External candidate structure universe (TRAIN + COCONUT + PubChem)

CPU / RAM heavy. **GPU is not used for Stage A**: RDKit parsing, tautomer canonicalization, canonical SMILES and InChIKey are CPU-bound, and a GPU substitute would risk changing the competition chemistry identity (`docs/STAGE_A_OPTIMIZATION.md`).

1. **Stage A** (optimized, `casmi.candidates.stage_a.run_stage_a`): the external source is copied once to the local SSD (size check), read in chunks (configured columns only) -> formula-mass prefilter -> the TRAINING canonicalizer (same calls; `universe_minimal` profile skips the EDA descriptors the universe never reads) on a persistent worker pool -> filters. Chunk outputs are written locally, persisted to Drive, and only then marked done. Markers carry a **build identity** (source file name + size, column mapping, chunk size, canonicalizer contract incl. RDKit version, filters): a changed source / config never reuses an old chunk.
2. **Stage B** (`run_stage_b`): per connectivity-key bucket dedup over the CURRENT complete Stage-A builds only; bucket markers carry a Stage-B identity -- a TRAIN-only bucket is rebuilt when COCONUT is added.
3. **Stage C** (`finalize_universe`): refuses buckets from another Stage-B identity; the manifest records sources, `external_source_present` and the build identity.

Resume after a disconnect: re-run from the top; finished chunks / buckets are skipped. Same code from a terminal or a big workstation: `scripts/build_external_universe.py`.

In [ ]:
import os, sys, json
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_ROOT = Path(os.environ.setdefault('ENVEDA_DRIVE_ROOT', '/content/drive/MyDrive/EnvedaCASMI'))
REPO_ROOT = Path(os.environ.setdefault('ENVEDA_REPO_ROOT', str(DRIVE_ROOT / 'repo')))
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from casmi.workspace.environment import ensure_packages, set_seeds, system_report
ensure_packages({'rdkit': 'rdkit', 'yaml': 'pyyaml'})
from casmi.workspace.config import load_v2_config, input_path
CFG, P = load_v2_config(REPO_ROOT / 'configs' / 'casmi_v2_colab.yaml')
P.ensure()
SEED = set_seeds(CFG['runtime']['seed'])
NOTEBOOK = '12_colab_candidate_universe'
print(system_report())

In [ ]:
import numpy as np, pandas as pd
from casmi.candidates.stage_a import StageAPerformance, configured_sources, stage_a_plan
from casmi.candidates.universe import UniverseBuildConfig
UCFG = UniverseBuildConfig.from_dict(CFG['universe'])                 # n_jobs: auto | integer override
PERF = StageAPerformance.from_dict(CFG['universe'].get('performance'))
OUT = P.candidate_db_dir / 'universe'       # buckets/, variants/, index/, formula_index/, universe_manifest.json
WORK = P.candidate_db_dir / 'universe_work' # stage_a/<build_id>/{standardized,rejected,filtered_out,_done}/ + stage_a/ACTIVE/
OUT.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)
# resolve every configured source; a present file is staged ONCE to the local SSD (reused when the size matches)
SOURCES = configured_sources(CFG, DRIVE_ROOT, REPO_ROOT, P.scratch_dir, PERF)
print(UCFG)
print(PERF)
with pd.option_context('display.max_colwidth', 120):
    display(stage_a_plan(SOURCES, UCFG, PERF).T)        # sizes, workers, GPU decision, chunk sizes, build id

In [ ]:
# TRAIN structures: the closed-world library keeps its training contract (representative SMILES, mass variants)
from casmi.candidates.filters import train_filter_audit
MV = pd.read_parquet(input_path(CFG, P, 'molecule_mass_variants'))
ST = pd.read_parquet(input_path(CFG, P, 'structure_table'), columns=['smiles', 'connectivity_key', 'molecular_weight', 'formal_charge'])
train_one = MV.sort_values(['connectivity_key', 'mass_variant_id']).drop_duplicates('connectivity_key')
train_one = train_one.merge(ST.dropna(subset=['connectivity_key']).drop_duplicates('connectivity_key')[['connectivity_key', 'formal_charge']],
                            on='connectivity_key', how='left')
print(f'TRAIN: {len(MV):,} mass variants, {MV.connectivity_key.nunique():,} connectivities')
print('filters TRAIN WOULD fail (diagnostic only -- TRAIN is never filtered by default):')
display(train_filter_audit(train_one, UCFG.filters))

In [ ]:
# Stage A - optimized, resumable per chunk (identity-checked markers; local temp -> Drive -> marker)
from casmi.candidates.stage_a import run_stage_a
STAGE_A = {}
for s in SOURCES:
    if not s.exists:
        print(f'SKIP {s.source}: {s.drive_path} not found (obtain it manually and upload it to Drive)')
        continue
    res = run_stage_a(s.cfg, WORK, P.scratch_dir, UCFG, PERF, source_identity_path=s.drive_path)
    STAGE_A[s.source] = res
    print(s.source, '| build', res.build_id, '| chunks', res.manifest['n_chunks'], '| input', res.manifest['n_input'], '| kept', res.manifest['n_kept'])
    display(res.summary[['chunk', 'n_input', 'n_prefiltered', 'n_rejected', 'n_filtered', 'n_kept', 'n_unique_raw_smiles',
                         'n_standardizer_calls', 'seconds']].tail(5))
if not STAGE_A:
    print('no external source processed -> the universe will contain TRAIN only (C2 external-universe protocol NOT valid)')

In [ ]:
# Stage B - per-bucket dedup by connectivity_key over the CURRENT complete Stage-A builds (identity-checked, resumable)
from tqdm.auto import tqdm
from casmi.candidates.stage_a import resolve_stage_a_inputs
from casmi.candidates.universe import run_stage_b
INPUTS = resolve_stage_a_inputs(WORK, SOURCES, UCFG, PERF)    # raises if a present source has no complete matching build
print({k: v.get('status') for k, v in INPUTS['sources'].items()})
ST_FOR_MERGE = ST.dropna(subset=['connectivity_key'])
STAGE_B = run_stage_b(INPUTS, MV, ST_FOR_MERGE, OUT, UCFG, progress=lambda b: tqdm(b, desc='buckets'))
BUCKETS = STAGE_B['buckets']
display(STAGE_B['records'].sort_values('n_connectivities', ascending=False).head(10))

In [ ]:
# Stage C - global candidate ids + mass index + formula index (only buckets of THIS Stage-B identity)
from casmi.candidates.universe import finalize_universe, universe_source_status
MANIFEST = finalize_universe(OUT, BUCKETS, stage_b_id=STAGE_B['stage_b_id'],
                             build_info={'stage_a': {k: v.get('build_id') for k, v in INPUTS['sources'].items()}})
print(json.dumps({k: MANIFEST[k] for k in ('n_candidates', 'n_buckets', 'external_source_present', 'n_train_candidates', 'n_external_only',
                                           'n_train_and_external', 'sources', 'build_id')}, indent=2))
UNIVERSE_STATUS = universe_source_status(OUT)
print('universe status:', UNIVERSE_STATUS['status'])
if UNIVERSE_STATUS['status'] != 'external':
    print('*** TRAIN-ONLY universe: a C2 external-universe evaluation is NOT valid until an external source (COCONUT) is built in.')

In [ ]:
# dedup statistics + source overlap (bucket by bucket: never the whole universe in RAM)
# pyarrow to_pylist for the list column: avoids the pandas-3 list-dtype metadata issue on read_parquet
import pyarrow.parquet as pq
from collections import Counter
overlap, n_records, n_conn = Counter(), 0, 0
for b in BUCKETS:
    t = pq.read_table(OUT / 'buckets' / f'bucket={b}.parquet', columns=['candidate_sources', 'n_source_records'])
    overlap.update('+'.join(sorted(s or [])) for s in t.column('candidate_sources').to_pylist())
    n_records += int(sum(x or 0 for x in t.column('n_source_records').to_pylist())); n_conn += t.num_rows
OVERLAP = pd.DataFrame(sorted(overlap.items(), key=lambda kv: -kv[1]), columns=['sources', 'n_connectivities'])
display(OVERLAP)
DEDUP = {'n_source_records_kept': n_records, 'n_unique_connectivities': n_conn,
         'records_per_connectivity': n_records / max(n_conn, 1)}
print(json.dumps(DEDUP, indent=2))
from casmi.candidates.universe import rejected_summary
REJ = rejected_summary(INPUTS['namespaces'])          # only the CURRENT Stage-A builds
display(REJ.sort_values('n', ascending=False).head(30))

In [ ]:
import matplotlib.pyplot as plt
from casmi.candidates.mass_index import CandidateMassIndex
from casmi.candidates.formula_index import CompactFormulaIndex
MIDX = CandidateMassIndex.load(OUT / 'index')
FIDX = CompactFormulaIndex.load(OUT / 'formula_index')
sample = np.asarray(MIDX.masses[:: max(1, len(MIDX) // 2_000_000)])   # strided sample of the mmap
fc = FIDX.counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].hist(sample, bins=120); axes[0].set_xlabel('exact mass (Da)'); axes[0].set_title(f'mass distribution ({len(MIDX):,} variant rows)')
axes[1].hist(np.log10(fc), bins=60); axes[1].set_xlabel('log10 #candidates per formula'); axes[1].set_title(f'{len(FIDX):,} formulas')
fig.tight_layout()
FIG = P.reports_dir / 'universe'; FIG.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG / 'universe_distributions.png', dpi=120)
top = np.argsort(-fc)[:20]
display(pd.DataFrame({'formula': FIDX.vocab[top], 'n_candidates': fc[top]}))

In [ ]:
from casmi.workspace.artifacts import write_metadata
from casmi.candidates.universe import UNIVERSE_SCHEMA_VERSION, V2_COLUMNS
mem = system_report()
write_metadata(OUT, NOTEBOOK, config={'universe': CFG['universe']}, seed=SEED,
               input_paths={'molecule_mass_variants': input_path(CFG, P, 'molecule_mass_variants'),
                            'structure_table': input_path(CFG, P, 'structure_table'),
                            **{s.source: s.drive_path for s in SOURCES}},
               schema_version=UNIVERSE_SCHEMA_VERSION, feature_list=V2_COLUMNS, repo_root=REPO_ROOT,
               extra={'manifest': MANIFEST, 'dedup': DEDUP, 'memory_at_end': mem, 'stage_a_builds': {k: r.build_id for k, r in STAGE_A.items()},
                      'stage_b_id': STAGE_B['stage_b_id'], 'universe_status': UNIVERSE_STATUS})
OVERLAP.to_parquet(FIG / 'source_overlap.parquet', index=False)
REJ.to_parquet(FIG / 'rejected_summary.parquet', index=False)
if STAGE_A:
    pd.concat([r.summary.drop(columns=['files'], errors='ignore') for r in STAGE_A.values()],
              ignore_index=True).to_parquet(FIG / 'stage_a_chunks.parquet', index=False)
print('universe saved under', OUT, '| memory:', mem)